# 單元二：Google Colab + AI 自動操作 Excel

> **學習目標：**
> - 能在 Google Colab 環境中上傳並讀取 Excel 檔案
> - 能用 pandas 完成篩選、統計、樞紐分析等操作
> - 能在 Colab 內直接呼叫 AI（Gemini）協助分析與產生程式
> - 能生成視覺化圖表並下載格式化報表

---

## Step 0：安裝套件 + 上傳檔案

先執行下方儲存格安裝必要套件，再上傳講師提供的 `銷售資料.xlsx`。

In [ ]:
# 安裝必要套件
!pip install openpyxl pandas xlsxwriter matplotlib

In [ ]:
from google.colab import files

# 會跳出檔案選擇視窗，請選擇講師提供的 Excel 檔案
# 建議一次上傳：銷售資料.xlsx、訂單.xlsx、產品.xlsx、成績單.xlsx、原始資料.xlsx
uploaded = files.upload()
print(f'已上傳 {len(uploaded)} 個檔案：')
for name in uploaded:
    print(f'  {name}')

---
## 2.3 用 Python 讀取 Excel

### 範例 11：讀取並顯示 Excel 內容

In [ ]:
import pandas as pd

# 讀取 Excel 檔案
df = pd.read_excel('銷售資料.xlsx')

# 顯示前 10 筆
print("=== 前 10 筆資料 ===")
print(df.head(10))

# 顯示基本資訊
print(f"\n共 {len(df)} 筆資料，{len(df.columns)} 個欄位")
print(f"欄位名稱：{list(df.columns)}")

### 範例 12：讀取特定工作表

如果一個 Excel 有多個工作表，可以指定要讀取哪一個。

> **注意：** 這個範例需要 `每月資料.xlsx`，如果還沒上傳請先上傳。

In [ ]:
# 如果需要上傳 每月資料.xlsx，取消下方註解執行
# from google.colab import files
# uploaded = files.upload()

# 讀取特定工作表
df_jan = pd.read_excel('每月資料.xlsx', sheet_name='1月')
print("=== 1月份資料 ===")
print(df_jan.head())

# 讀取所有工作表
all_sheets = pd.read_excel('每月資料.xlsx', sheet_name=None)
for sheet_name, data in all_sheets.items():
    print(f"工作表：{sheet_name}，共 {len(data)} 筆")

---
## 2.4 常見 Excel 操作的 Python 替代方案

### 範例 13：篩選資料（取代 Excel 篩選功能）

In [ ]:
import pandas as pd

df = pd.read_excel('銷售資料.xlsx')

# 篩選金額大於 10000 的訂單
big_orders = df[df['金額'] > 10000]
print("大額訂單：")
print(big_orders)

# 多條件篩選：業務員是王小明 且 產品是電子產品
filtered = df[(df['業務員'] == '王小明') & (df['產品類別'] == '電子產品')]
print("\n王小明的電子產品訂單：")
print(filtered)

# 篩選日期範圍
df['日期'] = pd.to_datetime(df['日期'])
jan_data = df[(df['日期'] >= '2024-01-01') & (df['日期'] <= '2024-01-31')]
print(f"\n一月份共 {len(jan_data)} 筆")

### 範例 14：樞紐分析表（取代 Excel Pivot Table）

In [ ]:
import pandas as pd

df = pd.read_excel('銷售資料.xlsx')

# 依業務員統計銷售金額（等同 Excel 樞紐分析表）
pivot = df.pivot_table(
    values='金額',
    index='業務員',
    columns='產品類別',
    aggfunc='sum',
    fill_value=0,
    margins=True  # 加上總計
)
print("=== 業務員 × 產品類別 銷售統計 ===")
print(pivot)

### 範例 15：VLOOKUP 的替代 — merge

> **需要檔案：** `訂單.xlsx` 和 `產品.xlsx`

In [ ]:
import pandas as pd

# 主表：訂單資料
orders = pd.read_excel('訂單.xlsx')
# 查詢表：產品價目表
products = pd.read_excel('產品.xlsx')

print("=== 訂單表（前 5 筆）===")
print(orders.head())
print("\n=== 產品表 ===")
print(products)

# 合併（等同 VLOOKUP）
result = orders.merge(products, on='產品編號', how='left')
print("\n=== 合併結果（前 10 筆）===")
print(result.head(10))

### 範例 16：條件新增欄位（取代 IF 公式）

> **需要檔案：** `成績單.xlsx`

In [ ]:
import pandas as pd

df = pd.read_excel('成績單.xlsx')

# 等同 Excel 的 IF 巢狀公式
def grade(score):
    if score >= 90: return 'A'
    elif score >= 80: return 'B'
    elif score >= 70: return 'C'
    elif score >= 60: return 'D'
    else: return 'F'

df['等第'] = df['分數'].apply(grade)
print(df)

### 範例 17：分組統計（取代 SUMIF / COUNTIF）

In [ ]:
import pandas as pd

df = pd.read_excel('銷售資料.xlsx')

# SUMIF 的替代：各業務員銷售總額
person_sum = df.groupby('業務員')['金額'].sum()
print("=== 各業務員銷售總額 ===")
print(person_sum)

# COUNTIF 的替代：各產品銷售筆數
product_count = df.groupby('產品類別')['金額'].count()
print("\n=== 各產品銷售筆數 ===")
print(product_count)

# AVERAGEIF 的替代：各業務員平均單筆金額
avg_sales = df.groupby('業務員')['金額'].mean().round(0)
print("\n=== 各業務員平均單筆金額 ===")
print(avg_sales)

### 範例 18：排序與排名（取代 RANK / 排序功能）

In [ ]:
import pandas as pd

df = pd.read_excel('銷售資料.xlsx')

# 依金額排序（由大到小）
df_sorted = df.sort_values('金額', ascending=False)
print("=== 銷售金額排行 ===")
print(df_sorted.head(10))

# 加上排名欄位
df['排名'] = df['金額'].rank(ascending=False, method='min').astype(int)
print(df[['業務員', '金額', '排名']].sort_values('排名'))

---
## 2.5 自動產生圖表（取代 Excel 圖表）

### 範例 19：自動產生長條圖與圓餅圖

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 設定中文字體（Colab 環境）
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK TC', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_excel('銷售資料.xlsx')

# 長條圖：各業務員銷售額
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sales_by_person = df.groupby('業務員')['金額'].sum().sort_values(ascending=False)
sales_by_person.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('各業務員銷售總額')
axes[0].set_ylabel('金額 (NT$)')

# 圓餅圖：各產品類別占比
sales_by_product = df.groupby('產品類別')['金額'].sum()
sales_by_product.plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('各產品類別銷售占比')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('銷售分析圖.png', dpi=150, bbox_inches='tight')
plt.show()
print("圖表已儲存為 銷售分析圖.png")

---
## 2.6 自動生成 Excel 報表並下載

### 範例 20：產生格式化 Excel 報表

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

df = pd.read_excel('銷售資料.xlsx')

# 建立統計資料
summary = df.groupby('業務員').agg(
    訂單數=('金額', 'count'),
    總金額=('金額', 'sum'),
    平均金額=('金額', 'mean'),
    最大單筆=('金額', 'max')
).round(0).reset_index()

# 建立格式化 Excel
wb = Workbook()
ws = wb.active
ws.title = "業務員績效報表"

# 標題
ws.merge_cells('A1:E1')
ws['A1'] = '2024 年度業務員績效報表'
ws['A1'].font = Font(size=16, bold=True, color='FFFFFF')
ws['A1'].fill = PatternFill(start_color='4472C4', fill_type='solid')
ws['A1'].alignment = Alignment(horizontal='center')

# 表頭
headers = ['業務員', '訂單數', '總金額', '平均金額', '最大單筆']
header_fill = PatternFill(start_color='D9E2F3', fill_type='solid')
for col, header in enumerate(headers, 1):
    cell = ws.cell(row=3, column=col, value=header)
    cell.font = Font(bold=True)
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center')

# 寫入資料
for r_idx, row in enumerate(dataframe_to_rows(summary, index=False, header=False), 4):
    for c_idx, value in enumerate(row, 1):
        cell = ws.cell(row=r_idx, column=c_idx, value=value)
        if c_idx >= 3:  # 金額欄位加千分位
            cell.number_format = '#,##0'

# 自動調整欄寬
for col in ws.columns:
    ws.column_dimensions[col[0].column_letter].width = 15

# 儲存
wb.save('業務員績效報表.xlsx')
print("報表已生成：業務員績效報表.xlsx")

# 在 Colab 中下載
from google.colab import files
files.download('業務員績效報表.xlsx')

### 範例 21：批次處理多個 Excel 檔案

> 此範例會讀取目前工作目錄中所有 `.xlsx` 檔案並合併。

In [ ]:
import pandas as pd
import glob

# 讀取資料夾中所有 Excel 檔案
all_files = glob.glob('*.xlsx')
print(f"找到 {len(all_files)} 個 Excel 檔案")

all_data = []
for file in all_files:
    try:
        df = pd.read_excel(file)
        df['來源檔案'] = file  # 標記來源
        all_data.append(df)
        print(f"  已讀取：{file}（{len(df)} 筆）")
    except Exception as e:
        print(f"  跳過：{file}（{e}）")

# 合併所有資料
if all_data:
    combined = pd.concat(all_data, ignore_index=True)
    print(f"\n合併完成，共 {len(combined)} 筆資料")

    # 儲存合併結果
    combined.to_excel('合併結果.xlsx', index=False)
    print("已儲存為：合併結果.xlsx")

### 範例 22：自動去除重複資料 & 清洗

> **需要檔案：** `原始資料.xlsx`（這份資料故意包含重複、空值、格式不一致的髒資料）

In [ ]:
import pandas as pd

df = pd.read_excel('原始資料.xlsx')
print(f"原始資料：{len(df)} 筆")

# 去除完全重複的列
df = df.drop_duplicates()
print(f"去除重複後：{len(df)} 筆")

# 去除空值
df = df.dropna(subset=['姓名', '金額'])  # 姓名和金額不可為空
print(f"去除空值後：{len(df)} 筆")

# 修正資料格式
df['姓名'] = df['姓名'].str.strip()       # 去除前後空白
df['電話'] = df['電話'].astype(str)         # 確保電話是文字
df['金額'] = pd.to_numeric(df['金額'], errors='coerce')  # 確保金額是數字

# 儲存清洗後的資料
df.to_excel('清洗後資料.xlsx', index=False)
print("清洗完成！已儲存。")
print(df.head(10))

---
## 2.7 在 Colab 內直接呼叫 AI 操作 Excel（Gemini）

### 使用 Colab 內建 AI 面板

在 Colab **右上角開啟 AI（Gemini）面板**，直接輸入以下需求：

```
請讀取目前工作目錄的 銷售資料.xlsx，
先告訴我這份資料的欄位與資料品質風險（空值、格式不一致），
再產生一段可執行的 pandas 程式：
1. 統計各產品類別銷售總額
2. 列出前 5 名業務員
3. 將結果輸出為 AI分析結果.xlsx
最後用繁體中文摘要 3 個發現。
```

### 範例 23（進階）：需要程式化串接時再使用 Gemini API

In [ ]:
# 先安裝 Google AI SDK
!pip install google-generativeai

In [ ]:
import pandas as pd
import google.generativeai as genai

# ============================================================
# 請在下方填入你的 API Key
# 取得方式：https://aistudio.google.com/apikey
# ============================================================
API_KEY = '請替換成你的_API_KEY'  # <-- 請修改這行
genai.configure(api_key=API_KEY)

# 讀取 Excel
df = pd.read_excel('銷售資料.xlsx')

# 把資料摘要傳給 AI 分析
data_summary = f"""
以下是銷售資料的摘要：

欄位：{list(df.columns)}
資料筆數：{len(df)}
數值欄位統計：
{df.describe().to_string()}

前 5 筆範例：
{df.head().to_string()}
"""

model = genai.GenerativeModel('gemini-2.0-flash')
response = model.generate_content(
    f"請用繁體中文分析以下銷售資料，給出 3 個重要發現和建議：\n{data_summary}"
)
print("=== AI 分析結果 ===")
print(response.text)

---
## 2.8 課堂練習（15 分鐘）

> **練習題：** 在 Colab 中完成以下任務：
> 1. 上傳任意一個 Excel 檔案，用 pandas 讀取並顯示前 10 筆
> 2. 使用 Colab 內建 AI（Gemini）產生 `groupby` 分組統計程式
> 3. 生成一個長條圖
> 4. 將結果儲存為新的 Excel 檔案並下載

### 自行練習區（請在下方撰寫程式）

In [ ]:
# 請在這裡撰寫你的練習程式
# 提示：
# 1. pd.read_excel('你的檔案.xlsx')
# 2. df.groupby('欄位名')['數值欄'].sum()
# 3. .plot(kind='bar')
# 4. df.to_excel('輸出.xlsx', index=False)

